In [0]:
!mkdir -p drive
!google-drive-ocamlfuse drive

/bin/bash: google-drive-ocamlfuse: command not found


In [0]:
import sys
sys.path.insert(0, 'drive/')

In [0]:
!pip install -q keras
!pip install -q tensorflow
!pip install -q numpy
!pip install -q pandas
!pip install -q nltk
!pip install -U -q PyDrive
!pip install -U -q sumeval


     |████████████████████████████████| 993kB 33.3MB/s 
     |████████████████████████████████| 51kB 15.9MB/s 


In [0]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# 1. Authenticate and create the PyDrive client.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# 2. Load a file by ID and create local file.
downloaded = drive.CreateFile({'id':'1RZ7L1ToT_f5cj73U26Dbi7sIm2V4xznF'}) # replace fileid with Id of file you want to access
downloaded.GetContentFile('wikihowAll.csv') # now you can use export.csv 

downloaded2 = drive.CreateFile({'id':'1_M0ya_yrrNwTEEPXjXycDM73KM8CR1Ln'}) # replace fileid with Id of file you want to access
downloaded2.GetContentFile('glove.6B.100d.txt') # now you can use export.csv 

In [0]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os as os
import re
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [0]:
from tensorflow.python.keras.models import Model
from tensorflow.python.keras.layers import Input,Dense,GRU,Embedding,CuDNNGRU,CuDNNLSTM
from tensorflow.python.keras.optimizers import RMSprop
from tensorflow.python.keras.callbacks import ModelCheckpoint
from tensorflow.python.keras.preprocessing.text import Tokenizer
from tensorflow.python.keras.preprocessing.sequence import pad_sequences
from sumeval.metrics.rouge import RougeCalculator
from nltk.translate.bleu_score import sentence_bleu

In [0]:
wikiHow = pd.read_csv("wikihowAll.csv")
#news

In [0]:
wikiHow.shape

(215365, 3)

In [0]:
wikiHow.head()

,headline,title,text
0,"\nKeep related supplies in the same area.,\nMa...",How to Be an Organized Artist1,"If you're a photographer, keep all the necess..."
1,\nCreate a sketch in the NeoPopRealist manner ...,How to Create a Neopoprealist Art Work,See the image for how this drawing develops s...
2,"\nGet a bachelor’s degree.,\nEnroll in a studi...",How to Be a Visual Effects Artist1,It is possible to become a VFX artist without...
3,\nStart with some experience or interest in ar...,How to Become an Art Investor,The best art investors do their research on t...
4,"\nKeep your reference materials, sketches, art...",How to Be an Organized Artist2,"As you start planning for a project or work, ..."


In [0]:
wikiHow.isnull().sum()

headline     818
title          1
text        1071
dtype: int64

In [0]:
wikiHow = wikiHow.dropna()
wikiHow = wikiHow.drop(['title'], 1)
wikiHow['text']=wikiHow['text'].fillna("")
wikiHow['headline']=wikiHow['headline'].fillna("")
wikiHow = wikiHow.reset_index(drop=True)

In [0]:
wikiHow.tail()

,headline,text
214289,\nConsider changing the spelling of your name....,"If you have a name that you like, you might f..."
214290,"\nTry out your name.,\nDon’t legally change yo...",Your name might sound great to you when you s...
214291,"\nUnderstand the process of relief printing.,\...",Relief printing is the oldest and most tradit...
214292,\nUnderstand the process of intaglio printing....,"Intaglio is Italian for ""incis­ing,"" and corr..."
214293,\nUnderstand the different varieties of lithog...,Lithography is a big term often used to refer...


In [0]:
wikiHow.text[15]

' Whether you are teaching a skill, delivering information or increasing awareness, outline the goals of your workshop. What do you want your workshop participants to learn? This analysis may result in a list of specific skills you will be teaching, concrete topics you will cover, or simply a feeling you will inspire in your participants. Think carefully about what you want to accomplish and why it is important.Some examples of workshop objectives include:\n\n\nLearn how to write a persuasive cover letter.\nLearn how to break bad news to a patient.\nLearn 5 techniques to get a reluctant student to talk in class.\nLearn how to create an effective Powerpoint presentation.;\n, Will the workshop participants know one another or are they strangers? Will they come in with knowledge about your topic or will they be completely unfamiliar with it? Are they choosing to attend your workshop or is it a requirement for their job training? Answers to all of these questions will affect how you organi

In [0]:
wikiHow.headline[15]

"\nDefine the workshop objective.,\nDecide who your audience is.,\nSchedule your workshop for the morning or early afternoon.,\nPublicize your workshop.,\nRecruit 8-15 participants for your workshop.,\nPrepare your participants for the workshop.,\nPrioritize your goals for the workshop.,\nPrepare a variety of teaching aids.,\nPrepare paper handouts.,\nArrange your audio-visual materials.,\nOrganize your computer-based materials.,\nRecruit experts, speakers, and assistants.,\nDecide on your group activities.,\nLeave time for breaks.,\nResist cramming.,\nSecure catering.,\nArrive early.,\nSet up all equipment before participants arrive.,\nArrange the chairs in advance.,\nDistribute materials.,\nGreet participants as they arrive.,\nIntroduce yourself and the workshop.,\nBegin icebreakers.,\nExecute your lesson plan.,\nBe flexible.,\nUse interactive exercises to reinforce information.,\nDon't talk too much.,\nStick to your scheduled breaks.,\nSwitch up activities every 20-30 minutes.,\nLig

In [0]:
contractions = { 
"ain't": "am not",
"aren't": "are not",
"can't": "cannot",
"can't've": "cannot have",
"'cause": "because",
"could've": "could have",
"couldn't": "could not",
"couldn't've": "could not have",
"didn't": "did not",
"doesn't": "does not",
"don't": "do not",
"hadn't": "had not",
"hadn't've": "had not have",
"hasn't": "has not",
"haven't": "have not",
"he'd": "he would",
"he'd've": "he would have",
"he'll": "he will",
"he's": "he is",
"how'd": "how did",
"how'll": "how will",
"how's": "how is",
"i'd": "i would",
"i'll": "i will",
"i'm": "i am",
"i've": "i have",
"isn't": "is not",
"it'd": "it would",
"it'll": "it will",
"it's": "it is",
"let's": "let us",
"ma'am": "madam",
"mayn't": "may not",
"might've": "might have",
"mightn't": "might not",
"must've": "must have",
"mustn't": "must not",
"needn't": "need not",
"oughtn't": "ought not",
"shan't": "shall not",
"sha'n't": "shall not",
"she'd": "she would",
"she'll": "she will",
"she's": "she is",
"should've": "should have",
"shouldn't": "should not",
"that'd": "that would",
"that's": "that is",
"there'd": "there had",
"there's": "there is",
"they'd": "they would",
"they'll": "they will",
"they're": "they are",
"they've": "they have",
"wasn't": "was not",
"we'd": "we would",
"we'll": "we will",
"we're": "we are",
"we've": "we have",
"weren't": "were not",
"what'll": "what will",
"what're": "what are",
"what's": "what is",
"what've": "what have",
"where'd": "where did",
"where's": "where is",
"who'll": "who will",
"who's": "who is",
"won't": "will not",
"wouldn't": "would not",
"you'd": "you would",
"you'll": "you will",
"you're": "you are"
}

In [0]:
def clean_text(text,contradictions = True):
    # Convert words to lower case
    
    if type(text) is str:
        text = text.lower()
    
    # Replace contractions with their longer forms 
    if contradictions:
        text = text.split()
        new_text = []
        for word in text:
            if word in contractions:
                new_text.append(contractions[word])
            else:
                new_text.append(word)
        text = " ".join(new_text)
    
    # Format words and remove unwanted characters
    text = re.sub(r'\<a href', ' ', text)
    text = re.sub(r'https?:\/\/.*[\r\n]*', '', text, flags=re.MULTILINE)
    text = re.sub(r'&amp;', '', text) 
    #text = re.sub(r'[_"\-;%()|+&=*%.,!?:#$@\[\]/]', ' ', text)
    text = re.sub(r'<br\s*\/?>', '', text)
    text = re.sub(r'<br />', ' ', text)
    text = re.sub(r'[^-''.,;<>\\|+!?"-*_a-zA-Z0-9 \n\.]', ' ', text)
    
    #text = re.sub(r'\'', ' ', text)

    return text

In [0]:
clean_summaries = []

for summary in wikiHow.headline:
    clean_summaries.append(clean_text(summary,contradictions = True))
print("Summaries are complete.")


Summaries are complete.


In [0]:
clean_texts = []
for text in wikiHow.text:
    clean_texts.append(clean_text(text,contradictions = True))
print("Texts are complete.")



Texts are complete.


In [0]:
clean_texts[15]

'whether you are teaching a skill, delivering information or increasing awareness, outline the goals of your workshop. what do you want your workshop participants to learn? this analysis may result in a list of specific skills you will be teaching, concrete topics you will cover, or simply a feeling you will inspire in your participants. think carefully about what you want to accomplish and why it is important.some examples of workshop objectives include  learn how to write a persuasive cover letter. learn how to break bad news to a patient. learn 5 techniques to get a reluctant student to talk in class. learn how to create an effective powerpoint presentation.; , will the workshop participants know one another or are they strangers? will they come in with knowledge about your topic or will they be completely unfamiliar with it? are they choosing to attend your workshop or is it a requirement for their job training? answers to all of these questions will affect how you organize your wo

In [0]:
clean_summaries[15]

'define the workshop objective., decide who your audience is., schedule your workshop for the morning or early afternoon., publicize your workshop., recruit 8-15 participants for your workshop., prepare your participants for the workshop., prioritize your goals for the workshop., prepare a variety of teaching aids., prepare paper handouts., arrange your audio-visual materials., organize your computer-based materials., recruit experts, speakers, and assistants., decide on your group activities., leave time for breaks., resist cramming., secure catering., arrive early., set up all equipment before participants arrive., arrange the chairs in advance., distribute materials., greet participants as they arrive., introduce yourself and the workshop., begin icebreakers., execute your lesson plan., be flexible., use interactive exercises to reinforce information., do not talk too much., stick to your scheduled breaks., switch up activities every 20-30 minutes., lighten the mood., maintain a res

In [0]:
mark_start = 'ssstarttoken '  #kelime haznesi içinde bulunmayan bir başlangıç tokeni veriyoruz.
#dekoder bu tokeni gördüğü zaman kelime üretmeye başlayacak.
mark_end = ' eeendtoken' #cümle bitiş tokeni dekoder cümlenin bitmesi karar verdiği zaman bu tokeni kullanacak
data_src = []
data_dest = []

data_src = clean_texts
for line in clean_summaries:
    data_dest.append(mark_start+line+mark_end)




In [0]:
print(data_src[100])
print(data_dest[100])


it can be any color, and any type, as long as it could not potentially harm your guinea pig. set it in the area you will be training in. be sure to turn it upside-down, so that your guinea pig will not knock it over as you begin to teach the trick. a garbage can that may have contained anything that could be harmful should not be used. , gently lift them out of their cage and set them down near the garbage can other object. allow the guinea pig to sniff around if they feel the need. some may ignore it, but this does not mean that they will not do the trick-have patience. , hold out one of the treats and lure your guinea pig towards the garbage can. move it slightly, further away and around it, until your guinea pig has completed a quarter of the circle. click and treat. do this three or four times, until your guinea pig becomes comfortable with moving around it a bit. , every time the guinea pig moves forward a few steps, reward them with a click(or verbal marker) and a treat. graduall

In [0]:
#len(data_src)
class TokenizerWrap(Tokenizer): #tokenizer nesnesini alıyoruz.
    def __init__(self,texts,padding,reverse=False,num_words=None):
        Tokenizer.__init__(self,num_words=num_words)
        
        self.fit_on_texts(texts) #tokenizera textlerimizi veriyoruz.
        self.index_to_word = dict(zip(self.word_index.values(),self.word_index.keys())) #kelime haznesindeki kelime rakam değerlerini key value olarak
        #yer değiştirir.
        self.tokens = self.texts_to_sequences(texts) #cümleler tokenlara çevrilir.
        
        if reverse:
            self.tokens = [list(reversed(x)) for x in self.tokens] #encodera vereceğimiz tüm inputlar aynı uzunlukta olmalı
            truncating = 'pre' #truncating işlemi cümlenin başından veya sonundan pre ve post durumuna göre token atar.
        else:
            truncating = 'post' #cümlenin başı sonundan daha önemli olduğundan gelen texti ters çeviriyoruz.
            
        self.num_tokens = [len(x) for x in self.tokens]
        self.max_tokens = np.mean(self.num_tokens) + 2 * np.std(self.num_tokens) #RNN e vereceğimiz vectörün boyutlarını belirliyoruz.
        #bu ortalama ve standart sapmayla optimum bir büyüklük belirliyoruz.
        self.max_tokens = int(self.max_tokens)
        
        self.tokens_padded = pad_sequences(self.tokens,  #gelen input textte sıfırdan oluşan padding ekleyip boyutları eşit hale getiriyoruz.
                                          maxlen = self.max_tokens,
                                          padding=padding,
                                          truncating = truncating)
        
    def token_to_word(self,token):#sayının kelime karşılığını döndürür
        word = ' ' if token == 0 else self.index_to_word[token]
        return word
    
    def tokens_to_string(self,tokens): #verilen tokenlardan cümle döndürür
        words = [self.index_to_word[token] for token in tokens if token != 0]
        text = ' '.join(words)
        return text
    
    def text_to_tokens(self,text,padding,reverse = False): #verilen texti tokenlara dönüştürür
        tokens = self.texts_to_sequences([text])
        tokens = np.array(tokens)
        
        if reverse:
            tokens = np.flip(tokens,axis=1)
            truncating = 'pre'
        else:
            truncating = 'post'
            
        tokens = pad_sequences(tokens,
                              maxlen=self.max_tokens,
                              padding=padding,
                              truncating = truncating)
        return tokens

                

In [0]:
tokenizer_src = TokenizerWrap(texts = data_src, #encodera verilecek 
                             padding = 'pre',
                             reverse = True,
                             num_words=None)

In [0]:
tokenizer_dest = TokenizerWrap(texts = data_dest,#decodere verilecek
                             padding = 'post',
                             reverse = False,
                             num_words=None)

In [0]:
tokens_src = tokenizer_src.tokens_padded
tokens_dest = tokenizer_dest.tokens_padded
print(tokens_src.shape)
print(tokens_dest.shape)

(214294, 1399)
(214294, 171)


In [0]:
tokens_dest[0]

array([    7,    53,  1252,   592,    10,     1,   256,   128,    21,
          26,  2776,     3,   103,     4,  4629,  3407,   104,   246,
        1886,    41,  1067,   592,    10,   202,  2175,  2398,  1706,
          19, 44098,     5,  2660,     3,   738,  5368,   766,     5,
        2141,   786,    19,   246,   404,     6,     1,   276,    11,
        1162,  1111,  1454,   334,    19, 10436,   230,     3,    21,
         334,    11, 10208,  1333,    61,    12,     1,  1267,   243,
           4,   987,  2369,     3,    21,     2,  1655,  2531,  3167,
        2711,    21,     4,  3623,     6,  3832,    29,   440,   888,
          14,  8685,  1207,    84,  1581,     8,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,

In [0]:
tokenizer_dest.tokens_to_string(tokens_dest[0])

'ssstarttoken keep related supplies in the same area make an effort to clean a dedicated workspace after every session place loose supplies in large clearly visible containers use clotheslines and clips to hang sketches photos and reference material use every inch of the room for storage especially vertical space use chalkboard paint to make space for drafting ideas right on the walls purchase a label maker to make your organization strategy semi permanent make a habit of throwing out old excess or useless stuff each month eeendtoken'

In [0]:
tokens_src[0]

array([ 0,  0,  0, ..., 15,  3, 14], dtype=int32)

In [0]:
tokenizer_src.tokens_to_string(tokens_src[0])

"it toss months six next the in it use will you chance little is there months six last the in it used not have you if here sentimental be not do sketch old an or paint right the find to junk through digging minutes 30 spending than fun more lot a is it but moment the at fun be not may it declutter to time aside set you if only but thing good a is this mess a making and experimenting things new making constantly are artists later for away it file or out it throw either project a of part or essential not is it if studio your of purge a do month a once art about think to mind your freeing labels the follow just can you things storing or for looking energy mental your of all spending of instead everything solve can maker label a with afternoon an but cleaning when uncertainty and items lost to leading effect opposite the has usually this frequently reorganizing by space your optimize to trying things of location the moving keep you when comes disorganization of lot a change needs your as c

In [0]:
token_start = tokenizer_dest.word_index[mark_start.strip()]
token_start

7

In [0]:
token_end = tokenizer_dest.word_index[mark_end.strip()]
token_end

8

In [0]:
encoder_input_data = tokens_src

In [0]:
decoder_input_data = tokens_dest[:,:-1] #sondan bir önceki tokena kadar verileri al
decoder_output_data = tokens_dest[:,1:] #1den sona kadar inputun 1 kaydırılmış hali

In [0]:
encoder_input_data[100]

array([ 0,  0,  0, ..., 16, 17, 11], dtype=int32)

In [0]:
decoder_input_data[100]

array([    7,    27,     4,   118,  4291,    46,    14,   911,  2733,
        1913,   955,     6,    62,  1521,    41,     2,  2370,  2296,
          10,     1,   886,   128,    19,     4,   484,     3,  4813,
           2,  2370,  2296,     3,    15,   544,    85,     5,   112,
          18,     1,  2370,  2296,    38,    56,     3,    43,   102,
          12,   288,   219,  4400,   283,   261,  4867,     2,  2370,
        2296,    40,    99,    21,     4,   511,   460,   102,     1,
         955,   283, 17707,     4,   172,  1062,    37,     1,  4291,
          46,   203,  1625,     8,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,

In [0]:
decoder_output_data[100]

array([   27,     4,   118,  4291,    46,    14,   911,  2733,  1913,
         955,     6,    62,  1521,    41,     2,  2370,  2296,    10,
           1,   886,   128,    19,     4,   484,     3,  4813,     2,
        2370,  2296,     3,    15,   544,    85,     5,   112,    18,
           1,  2370,  2296,    38,    56,     3,    43,   102,    12,
         288,   219,  4400,   283,   261,  4867,     2,  2370,  2296,
          40,    99,    21,     4,   511,   460,   102,     1,   955,
         283, 17707,     4,   172,  1062,    37,     1,  4291,    46,
         203,  1625,     8,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,

In [0]:
tokenizer_dest.tokens_to_string(decoder_input_data[100])

'ssstarttoken get a small garbage can or round cylinder shaped object of some sort place your guinea pig in the general area use a treat to lure your guinea pig to it stand back and see if the guinea pig will try to go around on its own eventually begin only treating your guinea pig when they make a full circle around the object begin enforcing a hand motion take the garbage can away gradually eeendtoken'

In [0]:
tokenizer_dest.tokens_to_string(decoder_output_data[100])

'get a small garbage can or round cylinder shaped object of some sort place your guinea pig in the general area use a treat to lure your guinea pig to it stand back and see if the guinea pig will try to go around on its own eventually begin only treating your guinea pig when they make a full circle around the object begin enforcing a hand motion take the garbage can away gradually eeendtoken'

In [0]:
num_encoder_words = len(tokenizer_src.word_index) + 1
num_decoder_words = len(tokenizer_dest.word_index) + 1

In [0]:
num_encoder_words

192102

In [0]:
num_decoder_words


89296

In [0]:
embedding_size = 100 #glove vektörlerinin uzunluğuyla aynı olmalı

In [0]:
word2vec = {} #eğitilmiş kelime glove kelime haznesini okuyoruz. Kelime anlamlarını tutuyor.
with open('glove.6B.100d.txt',encoding = 'UTF-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vec = np.array(values[1:], dtype='float32')
        word2vec[word] = vec


In [0]:
embedding_matrix = np.random.uniform(-1, 1, (num_encoder_words, embedding_size)) #önce boş bir matrix oluştur
for word,i in tokenizer_src.word_index.items():#kaynak textteki kelimeler glovedan gelen kelimeler içinde yoksa glovedan kelime alıyoruz.
    if i < num_encoder_words:
        embedding_vector = word2vec.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

In [0]:
embedding_matrix.shape


(192102, 100)

In [0]:
encoder_input = Input(shape=(None,),name='encoder_input')

In [0]:
encoder_embedding = Embedding(input_dim=num_encoder_words,
                             output_dim=embedding_size,
                             weights = [embedding_matrix],
                             trainable=True,
                             name='encoder_embedding')

In [0]:
state_size = 256

In [0]:
encoder_gru1 = CuDNNGRU(state_size,name="encoder_gru1",return_sequences=True)
encoder_gru2 = CuDNNGRU(state_size,name="encoder_gru2",return_sequences=True)
encoder_gru3 = CuDNNGRU(state_size,name="encoder_gru3",return_sequences=False)

In [0]:
def connect_encoder():
    layer = encoder_input
    layer = encoder_embedding(layer)
    layer = encoder_gru1(layer)
    layer = encoder_gru2(layer)
    layer = encoder_gru3(layer)
    
    encoder_output = layer
    
    return encoder_output

In [0]:
encoder_output = connect_encoder()

Instructions for updating:
Colocations handled automatically by placer.


In [0]:
decoder_initial_state = Input(shape = (state_size,),name='decoder_initial_state')

In [0]:
decoder_input = Input(shape=(None,),name='decoder_input')

In [0]:
decoder_embedding = Embedding(input_dim = num_decoder_words,
                             output_dim = embedding_size,
                             name = 'decoder_embedding')

In [0]:
decoder_gru1 = CuDNNGRU(state_size, name='decoder_gru1',return_sequences=True)
decoder_gru2 = CuDNNGRU(state_size, name='decoder_gru2',return_sequences=True)
decoder_gru3 = CuDNNGRU(state_size, name='decoder_gru3',return_sequences=True)

In [0]:
decoder_dense = Dense(num_decoder_words,
                     activation='linear',
                     name='decoder_output')

In [0]:
def connect_decoder(initial_state):
    layer = decoder_input
    layer = decoder_embedding(layer)
    layer = decoder_gru1(layer,initial_state = initial_state)
    layer = decoder_gru2(layer,initial_state = initial_state)
    layer = decoder_gru3(layer,initial_state = initial_state)
    
    decoder_output = decoder_dense(layer)
    
    return decoder_output

In [0]:
decoder_output = connect_decoder(initial_state=encoder_output)

In [0]:
model_train = Model(inputs=[encoder_input,decoder_input],outputs=[decoder_output])

In [0]:
model_encoder = Model(inputs=[encoder_input],outputs=[encoder_output])

In [0]:
decoder_output = connect_decoder(initial_state = decoder_initial_state)

In [0]:
model_decoder = Model(inputs=[decoder_input,decoder_initial_state],outputs=[decoder_output])

In [0]:
def sparse_cross_entropy(y_true,y_pred):
    loss = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y_true,logits=y_pred)
    loss_mean = tf.reduce_mean(loss)
    return loss_mean

In [0]:
optimizer = RMSprop(lr=1e-3)

In [0]:
decoder_target = tf.placeholder(dtype='int32',shape=(None,None))

In [0]:
model_train.compile(optimizer=optimizer,
                   loss=sparse_cross_entropy,
                   target_tensors=[decoder_target])

In [0]:
path_checkpoint = 'checkpoint.keras'
checkpoint = ModelCheckpoint(filepath=path_checkpoint,save_weights_only=True)

In [0]:
# create on Colab directory
model_train.save('model_more.h5')    
model_file = drive.CreateFile({'title' : 'model_more.h5'})
model_file.SetContentFile('model_more.h5')
model_file.Upload()

# download to google drive
drive.CreateFile({'id': model_file.get('id')})

model_train.save_weights('model_weights_more.h5')
weights_file = drive.CreateFile({'title' : 'model_weights_more.h5'})
weights_file.SetContentFile('model_weights_more.h5')
weights_file.Upload()
drive.CreateFile({'id': weights_file.get('id')})

GoogleDriveFile({'id': '1NJF4LUkimQgUnW1AcL6NdJMAZT1NEdS-'})

In [0]:



try:
  #  model_train.load_weights(path_checkpoint)
  # 3. reload weights from google drive into the model

  # use (get shareable link) to get file id
  last_weight_file = drive.CreateFile({'id': '1waobEqSnb2A5Cd0T_l5yC9Uo2Cwd6tNC'}) 
  last_weight_file.GetContentFile('last_weights_more.mat')
  model_train.load_weights('last_weights_more.mat')
except Exception as error:
    print('checkpoint yüklenemedi. eğitime baştan başlanıyor')
    print(error)
    

checkpoint yüklenemedi. eğitime baştan başlanıyor
Shapes (89296, 100) and (35609, 100) are incompatible


In [0]:
x_data = {'encoder_input': encoder_input_data,'decoder_input': decoder_input_data}


In [0]:
y_data = {'decoder_output': decoder_output_data}

In [0]:
model_train.fit(x=x_data,
               y=y_data,
               batch_size=64,
               epochs=1,
               callbacks=[checkpoint])
model_train.save('model_more.h5') 

214294/214294 [==============================] - 3875s 18ms/sample - loss: 1.5581


In [0]:
def CalculateRougeScores(refrence_summary,model_summary,scoring = False):
    rouge = RougeCalculator(stopwords=True, lang="en")
    rouge_1 = rouge.rouge_n(
            summary=model_summary,
            references=refrence_summary,
            n=1)
    rouge_2 = rouge.rouge_n(
            summary=model_summary,
            references=[refrence_summary],
            n=2)
    rouge_L = rouge.rouge_l(
            summary=model_summary,
            references=[refrence_summary])
    if (scoring == False):
        print("ROUGE-1: {}, ROUGE-2: {}, ROUGE-L: {}".format(
            rouge_1, rouge_2, rouge_L
        ).replace(", ", "\n"))
    
    return rouge_1,rouge_2,rouge_L

In [0]:
def CalculateBleuScores(refrence_summary,model_summary,scoring = False):
    refrence_summary_list = []
    rs =nltk.word_tokenize(refrence_summary)
    #print(r)
    refrence_summary_list.append(rs)
#     print(refrence_summary_list)
    model_summary = nltk.word_tokenize(model_summary)

#     print(model_summary)
#     a = [['good', 'quality', 'dog', 'food']]
#     b = ['a', 'good', 'dog', 'food']
    
    score = sentence_bleu(refrence_summary_list, model_summary,weights=(1, 0, 0, 0))
    if(scoring == False):
        print("BLEU Score: {} ".format(
            score,).replace(", ", "\n"))
        
    return score

In [0]:
def summarize(input_text, true_output_text=None,scoring = False):
    input_tokens = tokenizer_src.text_to_tokens(text=input_text,
                                                reverse=True,
                                                padding='pre')
    
    initial_state = model_encoder.predict(input_tokens)  
    max_tokens = tokenizer_dest.max_tokens    
    decoder_input_data = np.zeros(shape=(1, max_tokens), dtype=np.int)
    
        
    token_int = token_start
    output_text = ''
    count_tokens = 0
    
    while token_int != token_end and count_tokens < max_tokens:
        decoder_input_data[0, count_tokens] = token_int
        x_data = {'decoder_initial_state': initial_state, 'decoder_input': decoder_input_data}
        
        decoder_output = model_decoder.predict(x_data)
        
        token_onehot = decoder_output[0, count_tokens, :]
        token_int = np.argmax(token_onehot)
        
        sampled_word = tokenizer_dest.token_to_word(token_int)
        output_text += ' ' + sampled_word
        count_tokens += 1
        
    
       
    
    if scoring == False:
        print('Input text:')
        print(input_text)
        print()

        print('AI Summarized text:')
        print(output_text)
        print()
        if true_output_text is not None:
            print('Human Summarized text')
            print(true_output_text)
            print()
            
        rouge_1,rouge_2,rouge_L = CalculateRougeScores(true_output_text,output_text)
        score = CalculateBleuScores(true_output_text,output_text)
    else:
        rouge_1,rouge_2,rouge_L = CalculateRougeScores(true_output_text,output_text,True)
        score = CalculateBleuScores(true_output_text,output_text,True)  
    
    return rouge_1,rouge_2,rouge_L,score
    print()
    print()
    print()

In [0]:
def Scoring():    
    #pbar = ProgressBar()
    rouge_1_news = 0
    rouge_2_news = 0
    rouge_L_news = 0
    bleu_news = 0
    
    
    for i in range(1,500):
        print(i)
        rouge_1,rouge_2,rouge_L,bleu = summarize(clean_texts[i],clean_summaries[i],True)
        rouge_1_news = rouge_1_news + rouge_1
        rouge_2_news = rouge_2_news + rouge_2
        rouge_L_news = rouge_L_news + rouge_L
        bleu_news = bleu_news + bleu              
    
    rouge_1_news_avg_score = rouge_1_news / i
    rouge_2_news_avg_score = rouge_2_news / i
    rouge_L_news_avg_score = rouge_L_news / i
    bleu_news_avg_score = bleu_news / i
    
    
    print("rouge_1_news: {}, rouge_2_news: {}, rouge_L_news: {}, bleu_news: {}".
          format(rouge_1_news_avg_score,rouge_2_news_avg_score,rouge_L_news_avg_score,bleu_news_avg_score).replace(", ", "\n"))
    
    
    
    

In [1]:
summarize(input_text=data_src[60], true_output_text=data_dest[60])

NameError: ignored

In [0]:
Scoring()

1


/usr/local/lib/python3.6/dist-packages/nltk/translate/bleu_score.py:490: UserWarning: 
Corpus/Sentence contains 0 counts of 2-gram overlaps.
BLEU scores might be undesirable; use SmoothingFunction().
  warnings.warn(_msg)


2


/usr/local/lib/python3.6/dist-packages/nltk/translate/bleu_score.py:490: UserWarning: 
Corpus/Sentence contains 0 counts of 3-gram overlaps.
BLEU scores might be undesirable; use SmoothingFunction().
  warnings.warn(_msg)


3
4


/usr/local/lib/python3.6/dist-packages/nltk/translate/bleu_score.py:490: UserWarning: 
Corpus/Sentence contains 0 counts of 4-gram overlaps.
BLEU scores might be undesirable; use SmoothingFunction().
  warnings.warn(_msg)


5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
278
279
